# GridBrief: Forecasting GB grid carbon intensity and scheduling a deferrable load

End-to-end project that forecasts half-hourly GB grid carbon intensity from the NESO Carbon Intensity API and uses those forecasts to schedule a 2 kWh deferrable load at the lowest-carbon window.

**Design principle:** the method is selected on a held-out validation period (October 2025) and frozen before the test set (Nov to Dec 2025) is evaluated. No test leakage into model selection.

**Headline finding:** Ridge Regression had the lowest test MAE (40.83 gCO2/kWh), but no forecast method beat a trivial fixed 02:00-04:00 off-peak rule on this grid over this test period. The negative result is itself the point of the project.

## 0. Setup: paths and directories

Project directories are created once so any later cell can be run in order.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import requests
from IPython.display import display as show_table

PROJECT_DIR = Path("/content/gridbrief")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
IMAGES_DIR = PROJECT_DIR / "images"

for d in (RAW_DIR, PROCESSED_DIR, RESULTS_DIR, IMAGES_DIR):
    d.mkdir(parents=True, exist_ok=True)

TARGET = "carbon_intensity_gco2_per_kwh"
print("Project directories ready:")
for d in (RAW_DIR, PROCESSED_DIR, RESULTS_DIR, IMAGES_DIR):
    print(" -", d)

## 1. Data download

A single-day sample confirms the API contract, then the full 2025 year is downloaded in 14-day batches with disk caching and retry logic.

In [ ]:
sample_date = "2025-01-01"
url = f"https://api.carbonintensity.org.uk/intensity/date/{sample_date}"
response = requests.get(url, headers={"Accept": "application/json"}, timeout=(10, 30))
response.raise_for_status()

sample_payload = response.json()
sample_records = sample_payload.get("data", [])
if not sample_records:
    raise ValueError("The data service returned no records.")

sample_path = RAW_DIR / f"sample_{sample_date}.json"
sample_path.write_text(json.dumps(sample_payload, indent=2), encoding="utf-8")

sample_df = pd.json_normalize(sample_records).rename(columns={"intensity.actual": TARGET})
for column in ["from", "to"]:
    sample_df[column] = pd.to_datetime(sample_df[column], utc=True)

print("Dataset connection successful.")
print("Half-hourly records:", len(sample_df))
print("Saved:", sample_path)
show_table(sample_df[["from", "to", TARGET]].head())

In [ ]:
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

START = pd.Timestamp("2025-01-01", tz="UTC")
END = pd.Timestamp("2026-01-01", tz="UTC")
batch_starts = pd.date_range(START, END, freq="14D", inclusive="left")

BATCH_DIR = RAW_DIR / "2025_batches"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=2, backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)))

frames = []
for number, batch_start in enumerate(batch_starts, start=1):
    batch_end = min(batch_start + pd.Timedelta(days=14), END)
    start_text = batch_start.strftime("%Y-%m-%dT%H:%MZ")
    end_text = batch_end.strftime("%Y-%m-%dT%H:%MZ")
    url = f"https://api.carbonintensity.org.uk/intensity/{start_text}/{end_text}"
    cache_path = BATCH_DIR / f"{batch_start:%Y%m%d}_{batch_end:%Y%m%d}.json"

    if cache_path.exists():
        saved_batch = json.loads(cache_path.read_text(encoding="utf-8"))
        source = "reused"
    else:
        response = session.get(url, headers={"Accept": "application/json"}, timeout=(10, 60))
        response.raise_for_status()
        payload = response.json()
        if not payload.get("data"):
            raise ValueError(f"No records returned for {start_text}.")
        saved_batch = {
            "request_url": url,
            "downloaded_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            "response": payload,
        }
        temporary_path = cache_path.with_suffix(".tmp")
        temporary_path.write_text(json.dumps(saved_batch, indent=2), encoding="utf-8")
        temporary_path.replace(cache_path)
        source = "downloaded"
        time.sleep(0.5)

    batch_df = pd.json_normalize(saved_batch["response"]["data"])
    for column in ["from", "to"]:
        batch_df[column] = pd.to_datetime(batch_df[column], utc=True)
    batch_df = batch_df.loc[(batch_df["from"] >= batch_start) & (batch_df["from"] < batch_end)].copy()
    if batch_df.empty:
        raise ValueError(f"No matching intervals for {start_text}.")
    frames.append(batch_df)
    print(f"{number:02}/{len(batch_starts)} | {batch_start:%Y-%m-%d} to {batch_end:%Y-%m-%d} | {len(batch_df)} rows | {source}", flush=True)

df_raw = (
    pd.concat(frames, ignore_index=True)
    .rename(columns={
        "intensity.actual": TARGET,
        "intensity.forecast": "provider_forecast_gco2_per_kwh",
        "intensity.index": "provider_index",
    })
    .sort_values("from")
    .reset_index(drop=True)
)

dataset_path = RAW_DIR / "gb_carbon_intensity_2025.csv"
df_raw.to_csv(dataset_path, index=False)
session.close()

print("\nDownload complete.")
print(f"Total rows: {len(df_raw):,}")
print("First interval starts:", df_raw["from"].min())
print("Last interval ends:", df_raw["to"].max())
print("Dataset saved:", dataset_path)

## 2. Data quality audit

Confirm continuous half-hourly coverage, no missing or duplicate timestamps, and clean numeric target values.

In [ ]:
expected_times = pd.date_range(START, END, freq="30min", inclusive="left")
observed_times = pd.DatetimeIndex(df_raw["from"].dropna())
missing_times = expected_times.difference(observed_times)
unexpected_times = observed_times.difference(expected_times)

numeric_values = pd.to_numeric(df_raw[TARGET], errors="coerce")
non_numeric = df_raw[TARGET].notna() & numeric_values.isna()
durations = df_raw["to"] - df_raw["from"]

audit = {
    "Expected half-hourly intervals": len(expected_times),
    "Downloaded rows": len(df_raw),
    "Rows missing start or end timestamps": int(df_raw[["from", "to"]].isna().any(axis=1).sum()),
    "Duplicate start timestamps": int(df_raw["from"].duplicated().sum()),
    "Missing half-hourly intervals": len(missing_times),
    "Unexpected start timestamps": len(unexpected_times),
    "Intervals with incorrect duration": int(durations.ne(pd.Timedelta(minutes=30)).sum()),
    "Missing carbon-intensity values": int(df_raw[TARGET].isna().sum()),
    "Non-numeric carbon-intensity values": int(non_numeric.sum()),
    "Infinite carbon-intensity values": int(np.isinf(numeric_values).sum()),
    "Negative carbon-intensity values": int(numeric_values.lt(0).sum()),
}

audit_report = pd.Series(audit, name="count").rename_axis("check")
audit_report.to_csv(RESULTS_DIR / "data_quality_audit.csv")
pd.DataFrame({"missing_interval_start_utc": missing_times}).to_csv(RESULTS_DIR / "missing_intervals.csv", index=False)

print("DATA QUALITY AUDIT\n")
print(audit_report.to_string())
print("\nMISSING VALUES BY COLUMN\n")
print(df_raw.isna().sum().to_string())

## 3. Prepare analysis dataset

In [ ]:
df_clean = (
    df_raw[["from", "to", TARGET]]
    .copy()
    .rename(columns={"from": "timestamp_utc", "to": "interval_end_utc"})
)
for column in ["timestamp_utc", "interval_end_utc"]:
    df_clean[column] = pd.to_datetime(df_clean[column], utc=True)
df_clean[TARGET] = pd.to_numeric(df_clean[TARGET], errors="raise").astype("float64")
df_clean = df_clean.sort_values("timestamp_utc").reset_index(drop=True)

processed_path = PROCESSED_DIR / "gb_carbon_intensity_2025.csv"
df_clean.to_csv(processed_path, index=False)

print("Analysis dataset prepared.")
print(f"Rows retained: {len(df_clean):,}")
print("Missing target values:", int(df_clean[TARGET].isna().sum()))
print("Saved:", processed_path)

## 4. Exploratory charts

Daily trend, monthly distribution, and weekday-time heatmap.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

daily = df_clean.set_index("timestamp_utc")[TARGET].resample("D").mean().to_frame(name="daily_mean")
daily["seven_day_mean"] = daily["daily_mean"].rolling(window=7, min_periods=7).mean()
daily.to_csv(RESULTS_DIR / "daily_carbon_intensity_2025.csv", index_label="date_utc")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily.index, daily["daily_mean"], color="#8CBCAF", linewidth=1.2, label="Daily average")
ax.plot(daily.index, daily["seven_day_mean"], color="#0F766E", linewidth=2.5, label="7-day trailing average")
ax.set_title("GridBrief: daily carbon intensity in Great Britain, 2025", fontsize=15, loc="left", pad=16)
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Carbon intensity (gCO2/kWh)")
ax.set_xlim(daily.index.min(), daily.index.max())
ax.set_ylim(bottom=0)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.grid(axis="y", color="#E5E7EB", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=2)
fig.text(0.01, 0.01, "Source: NESO Carbon Intensity API | Historical national estimates", fontsize=9, color="#64748B")
fig.tight_layout(rect=[0, 0.04, 1, 1])
chart_path = IMAGES_DIR / "01_daily_carbon_intensity.png"
fig.savefig(chart_path, dpi=180, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig)
print("Chart saved:", chart_path)

In [ ]:
import calendar

monthly_data = df_clean.assign(month=df_clean["timestamp_utc"].dt.month)
month_numbers = list(range(1, 13))
month_names = [calendar.month_abbr[m] for m in month_numbers]
monthly_values = [monthly_data.loc[monthly_data["month"] == m, TARGET].to_numpy() for m in month_numbers]
monthly_summary = (
    monthly_data.groupby("month")[TARGET]
    .agg(["count", "mean", "median", "min", "max"])
    .reindex(month_numbers)
)
monthly_summary.index = pd.Index(month_names, name="month")
monthly_summary.to_csv(RESULTS_DIR / "monthly_carbon_intensity_summary.csv")

fig, ax = plt.subplots(figsize=(12, 5))
box_artists = ax.boxplot(
    monthly_values, positions=month_numbers, widths=0.6, patch_artist=True,
    medianprops={"color": "#0F172A", "linewidth": 1.8},
    whiskerprops={"color": "#64748B"}, capprops={"color": "#64748B"},
    flierprops={"marker": "o", "markersize": 2, "markerfacecolor": "#64748B", "markeredgecolor": "none", "alpha": 0.3},
)
for box in box_artists["boxes"]:
    box.set_facecolor("#8CBCAF"); box.set_edgecolor("#0F766E")
ax.set_title("GridBrief: carbon-intensity distributions by month, 2025", fontsize=15, loc="left", pad=16)
ax.set_xticks(month_numbers, labels=month_names)
ax.set_xlabel("Month (UTC)"); ax.set_ylabel("Carbon intensity (gCO2/kWh)")
ax.set_ylim(bottom=0)
ax.grid(axis="y", color="#E5E7EB", linewidth=0.8); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, 0.01, "Source: NESO Carbon Intensity API | Half-hourly national estimates", fontsize=9, color="#64748B")
fig.tight_layout(rect=[0, 0.04, 1, 1])
chart_path = IMAGES_DIR / "02_monthly_distribution.png"
fig.savefig(chart_path, dpi=180, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig)
show_table(monthly_summary.round(1))
print("Chart saved:", chart_path)

In [ ]:
profile_data = df_clean.assign(
    weekday=df_clean["timestamp_utc"].dt.dayofweek,
    half_hour_slot=df_clean["timestamp_utc"].dt.hour * 2 + df_clean["timestamp_utc"].dt.minute // 30,
)
weekday_profile = (
    profile_data.pivot_table(index="weekday", columns="half_hour_slot", values=TARGET, aggfunc="mean")
    .reindex(index=range(7), columns=range(48))
)
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
time_labels = [f"{slot // 2:02d}:{(slot % 2) * 30:02d}" for slot in range(48)]
weekday_profile.index = pd.Index(day_names, name="weekday_utc")
weekday_profile.columns = time_labels
weekday_profile.to_csv(RESULTS_DIR / "weekday_time_profile.csv")

fig, ax = plt.subplots(figsize=(12, 5))
heatmap = ax.imshow(weekday_profile.to_numpy(), aspect="auto", interpolation="nearest", cmap="YlGnBu")
tick_slots = list(range(0, 48, 4))
ax.set_xticks(tick_slots, labels=[time_labels[s] for s in tick_slots])
ax.set_yticks(range(7), labels=day_names)
ax.set_title("GridBrief: average carbon intensity by weekday and time, 2025", fontsize=15, loc="left", pad=16)
ax.set_xlabel("Half-hour interval start (UTC)"); ax.set_ylabel("Weekday (UTC)")
cb = fig.colorbar(heatmap, ax=ax, pad=0.02); cb.set_label("Average carbon intensity (gCO2/kWh)")
fig.text(0.01, 0.01, "Source: NESO Carbon Intensity API | Means across 2025 observations", fontsize=9, color="#64748B")
fig.tight_layout(rect=[0, 0.04, 1, 1])
chart_path = IMAGES_DIR / "03_weekday_time_heatmap.png"
fig.savefig(chart_path, dpi=180, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig)
print("Chart saved:", chart_path)

## 5. Feature engineering

Lag features (1, 2, 7 days), previous-day statistics, and cyclical encodings. The first 7 days are used only as history.

In [ ]:
frame = df_clean.copy()
times = frame["timestamp_utc"]
frame["forecast_origin_utc"] = times.dt.floor("D")

lag_steps = {"lag_1_day": 48, "lag_2_days": 96, "lag_7_days": 336}
for name, steps in lag_steps.items():
    frame[name] = frame[TARGET].shift(steps)
    source_end = frame["interval_end_utc"].shift(steps)
    assert (source_end.isna() | source_end.le(frame["forecast_origin_utc"])).all(), f"Timing check failed for {name}."

daily_history = (
    df_clean.set_index("timestamp_utc")[TARGET]
    .resample("D").agg(["mean", "std"]).shift(1)
    .rename(columns={"mean": "previous_day_mean", "std": "previous_day_std"})
)
frame = frame.join(daily_history, on="forecast_origin_utc")

cycles = {
    "hour": (times.dt.hour + times.dt.minute / 60, 24),
    "weekday": (times.dt.dayofweek, 7),
    "year": (times.dt.dayofyear - 1, np.where(times.dt.is_leap_year, 366, 365)),
}
for name, (values, period) in cycles.items():
    frame[f"{name}_sin"] = np.sin(2 * np.pi * values / period)
    frame[f"{name}_cos"] = np.cos(2 * np.pi * values / period)
frame["is_weekend"] = (times.dt.dayofweek >= 5).astype(int)

FEATURE_COLUMNS = [
    *lag_steps,
    "previous_day_mean", "previous_day_std",
    "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos",
    "year_sin", "year_cos",
    "is_weekend",
]

df_model = frame.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
assert np.isfinite(df_model[FEATURE_COLUMNS].to_numpy()).all()
assert df_model.groupby("forecast_origin_utc").size().eq(48).all()

df_model.to_csv(PROCESSED_DIR / "model_dataset_2025.csv", index=False)
(RESULTS_DIR / "forecast_setup.json").write_text(json.dumps({
    "forecast_origin": "00:00 UTC each day",
    "forecast_intervals": 48,
    "interval_minutes": 30,
    "target": TARGET,
    "features": FEATURE_COLUMNS,
    "lag_steps": lag_steps,
}, indent=2), encoding="utf-8")

print(f"Modeling rows: {len(df_model):,}")
print(f"First forecast day: {df_model['forecast_origin_utc'].min().date()}")
print(f"Last forecast day:  {df_model['forecast_origin_utc'].max().date()}")

## 6. Train / validation / test split

October 2025 = validation (method selection). Nov to Dec 2025 = test (evaluated once, frozen).

In [ ]:
validation_start = pd.Timestamp("2025-10-01", tz="UTC")
test_start = pd.Timestamp("2025-11-01", tz="UTC")
origins = df_model["forecast_origin_utc"]

train_df = df_model.loc[origins < validation_start].copy()
val_df = df_model.loc[(origins >= validation_start) & (origins < test_start)].copy()
test_df = df_model.loc[origins >= test_start].copy()

assert len(train_df) + len(val_df) + len(test_df) == len(df_model)
assert train_df["forecast_origin_utc"].max() < val_df["forecast_origin_utc"].min()
assert val_df["forecast_origin_utc"].max() < test_df["forecast_origin_utc"].min()
assert TARGET not in FEATURE_COLUMNS

X_train, y_train = train_df[FEATURE_COLUMNS].copy(), train_df[TARGET].copy()
X_val, y_val = val_df[FEATURE_COLUMNS].copy(), val_df[TARGET].copy()
X_test, y_test = test_df[FEATURE_COLUMNS].copy(), test_df[TARGET].copy()

split_summary = pd.DataFrame([
    {"set": name, "rows": len(part),
     "forecast_days": len(part.groupby("forecast_origin_utc")),
     "first_day": part["forecast_origin_utc"].min().date(),
     "last_day": part["forecast_origin_utc"].max().date()}
    for name, part in (("Training", train_df), ("Validation", val_df), ("Test", test_df))
])
split_summary.to_csv(RESULTS_DIR / "split_summary.csv", index=False)
show_table(split_summary)

## 7. Validation: three forecast methods

Same-time-yesterday baseline, Ridge Regression, and Histogram Gradient Boosting, all evaluated on October 2025.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_val_pred = X_val["lag_1_day"].to_numpy(copy=True)
baseline_val_metrics = pd.DataFrame([{
    "model": "Same time yesterday",
    "mae_gco2_per_kwh": mean_absolute_error(y_val, baseline_val_pred),
    "rmse_gco2_per_kwh": np.sqrt(mean_squared_error(y_val, baseline_val_pred)),
}])
baseline_val_metrics.to_csv(RESULTS_DIR / "baseline_validation_metrics.csv", index=False)

baseline_val_rows = val_df[["timestamp_utc", "forecast_origin_utc", TARGET]].copy().rename(columns={TARGET: "actual_gco2_per_kwh"})
baseline_val_rows["predicted_gco2_per_kwh"] = baseline_val_pred
baseline_val_rows.to_csv(RESULTS_DIR / "baseline_validation_predictions.csv", index=False)
show_table(baseline_val_metrics.round(3))

In [ ]:
from time import perf_counter
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge_model = Pipeline([("scaler", StandardScaler()), ("regressor", Ridge(alpha=1.0))])
t0 = perf_counter(); ridge_model.fit(X_train, y_train); ridge_training_seconds = perf_counter() - t0
ridge_val_pred = ridge_model.predict(X_val)

ridge_val_metrics = pd.DataFrame([{
    "model": "Ridge Regression",
    "mae_gco2_per_kwh": mean_absolute_error(y_val, ridge_val_pred),
    "rmse_gco2_per_kwh": np.sqrt(mean_squared_error(y_val, ridge_val_pred)),
}])
ridge_val_metrics.to_csv(RESULTS_DIR / "ridge_validation_metrics.csv", index=False)
print(f"Ridge training time: {ridge_training_seconds:.2f} s")
show_table(ridge_val_metrics.round(3))

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_model = HistGradientBoostingRegressor(
    loss="absolute_error", learning_rate=0.05, max_iter=200,
    max_leaf_nodes=15, min_samples_leaf=30,
    early_stopping=False, random_state=42,
)
print("Training Histogram Gradient Boosting...")
t0 = perf_counter(); hgb_model.fit(X_train, y_train); hgb_training_seconds = perf_counter() - t0
hgb_val_pred = hgb_model.predict(X_val)

hgb_val_metrics = pd.DataFrame([{
    "model": "Histogram Gradient Boosting",
    "mae_gco2_per_kwh": mean_absolute_error(y_val, hgb_val_pred),
    "rmse_gco2_per_kwh": np.sqrt(mean_squared_error(y_val, hgb_val_pred)),
}])
hgb_val_metrics.to_csv(RESULTS_DIR / "hgb_validation_metrics.csv", index=False)
print(f"Training time: {hgb_training_seconds:.2f} s | rounds: {hgb_model.n_iter_}")

validation_comparison = (
    pd.concat([baseline_val_metrics, ridge_val_metrics, hgb_val_metrics], ignore_index=True)
    .sort_values("mae_gco2_per_kwh", kind="stable").reset_index(drop=True)
)
validation_comparison.to_csv(RESULTS_DIR / "validation_model_comparison.csv", index=False)
show_table(validation_comparison.round(3))

## 8. Method selection (validation) and one-shot test evaluation

In [ ]:
best_validation_row = validation_comparison.sort_values("mae_gco2_per_kwh", kind="stable").iloc[0]
selected_method = str(best_validation_row["model"])

selection_record = {
    "selected_method": selected_method,
    "selection_criterion": "Lowest validation MAE",
    "validation_mae_gco2_per_kwh": float(best_validation_row["mae_gco2_per_kwh"]),
    "validation_start_utc": val_df["timestamp_utc"].min().isoformat(),
    "validation_end_utc": val_df["interval_end_utc"].max().isoformat(),
    "refit_after_validation": False,
}
(RESULTS_DIR / "model_selection.json").write_text(json.dumps(selection_record, indent=2), encoding="utf-8")

test_predictions = {
    "Same time yesterday": X_test["lag_1_day"].to_numpy(copy=True),
    "Ridge Regression": ridge_model.predict(X_test),
    "Histogram Gradient Boosting": hgb_model.predict(X_test),
}
test_prediction_rows = test_df[["timestamp_utc", "forecast_origin_utc", TARGET]].copy().rename(columns={TARGET: "actual_gco2_per_kwh"})

test_metric_rows = []
for name, preds in test_predictions.items():
    test_prediction_rows[name] = preds
    test_metric_rows.append({
        "model": name,
        "mae_gco2_per_kwh": mean_absolute_error(y_test, preds),
        "rmse_gco2_per_kwh": np.sqrt(mean_squared_error(y_test, preds)),
        "selected_on_validation": name == selected_method,
    })

test_comparison = pd.DataFrame(test_metric_rows).sort_values("mae_gco2_per_kwh", kind="stable").reset_index(drop=True)
selected_test_pred = test_predictions[selected_method].copy()
test_comparison.to_csv(RESULTS_DIR / "test_model_comparison.csv", index=False)
test_prediction_rows.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

print("Selected using validation:", selected_method)
show_table(test_comparison.round(3))

## 9. Test-set charts: model comparison and first-week forecast vs actual

In [ ]:
MODEL_COLORS = {
    "Same time yesterday": "#64748B",
    "Ridge Regression": "#2563EB",
    "Histogram Gradient Boosting": "#0F766E",
}
plot_data = test_comparison.copy()
positions = np.arange(len(plot_data))
labels = [f"{n}\n(selected on validation)" if n == selected_method else n for n in plot_data["model"]]
colors = [MODEL_COLORS[n] for n in plot_data["model"]]
panels = [("mae_gco2_per_kwh", "Mean absolute error"), ("rmse_gco2_per_kwh", "Root mean squared error")]
axis_limit = 1.2 * plot_data[["mae_gco2_per_kwh", "rmse_gco2_per_kwh"]].to_numpy().max()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (column, title) in zip(axes, panels):
    bars = ax.barh(positions, plot_data[column], color=colors, height=0.55)
    ax.set_yticks(positions, labels=labels)
    ax.set_xlim(0, axis_limit)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Error (gCO2/kWh); lower is better")
    ax.bar_label(bars, fmt="%.2f", padding=5, fontsize=10)
    ax.set_axisbelow(True); ax.grid(axis="x", alpha=0.2)
    for side in ("top", "right"): ax.spines[side].set_visible(False)
axes[0].invert_yaxis()
fig.suptitle("GridBrief: forecasting errors on the test period", fontsize=15, fontweight="bold")
fig.text(0.02, 0.025, "Source: NESO Carbon Intensity API | Nov to Dec 2025 | 2,928 half-hour intervals | Selection based on October validation MAE", fontsize=9, color="#64748B")
fig.tight_layout(rect=[0, 0.09, 1, 0.92])
chart_path = IMAGES_DIR / "04_test_model_comparison.png"
fig.savefig(chart_path, dpi=200, bbox_inches="tight"); plt.show(); plt.close(fig)
print("Chart saved:", chart_path)

In [ ]:
week_start = test_prediction_rows["timestamp_utc"].min().floor("D")
week_end = week_start + pd.Timedelta(days=7)
week_mask = (test_prediction_rows["timestamp_utc"] >= week_start) & (test_prediction_rows["timestamp_utc"] < week_end)
first_week = (
    test_prediction_rows.loc[week_mask, ["timestamp_utc", "forecast_origin_utc", "actual_gco2_per_kwh", selected_method]]
    .rename(columns={selected_method: "predicted_gco2_per_kwh"})
    .sort_values("timestamp_utc").copy()
)
assert len(first_week) == 336
week_mae = mean_absolute_error(first_week["actual_gco2_per_kwh"], first_week["predicted_gco2_per_kwh"])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(first_week["timestamp_utc"], first_week["actual_gco2_per_kwh"], color="#0F766E", linewidth=2, label="Historical estimate")
ax.plot(first_week["timestamp_utc"], first_week["predicted_gco2_per_kwh"], color=MODEL_COLORS[selected_method], linewidth=1.8, linestyle="--", label=f"Forecast: {selected_method}")
ax.set_xlim(week_start, week_end); ax.set_ylim(bottom=0)
ax.set_xlabel("Date (UTC)"); ax.set_ylabel("Carbon intensity (gCO2/kWh)")
ax.xaxis.set_major_locator(mdates.DayLocator(tz="UTC")); ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b", tz="UTC"))
ax.grid(alpha=0.2); ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncol=2, frameon=False)
for side in ("top", "right"): ax.spines[side].set_visible(False)
fig.suptitle("GridBrief: selected forecasts versus historical estimates", fontsize=15, fontweight="bold")
fig.text(0.02, 0.025, f"Source: NESO Carbon Intensity API | 1 to 7 November 2025 | First-week MAE: {week_mae:.2f} gCO2/kWh", fontsize=9, color="#64748B")
fig.tight_layout(rect=[0, 0.09, 1, 0.93])
chart_path = IMAGES_DIR / "05_first_test_week_forecast.png"
fig.savefig(chart_path, dpi=200, bbox_inches="tight"); plt.show(); plt.close(fig)
first_week.to_csv(RESULTS_DIR / "first_test_week_predictions.csv", index=False)
print(f"Half-hour intervals plotted: {len(first_week)}")
print("Forecast method:", selected_method)
print(f"First-week MAE: {week_mae:.3f} gCO2/kWh")
print("Chart saved:", chart_path)

## 10. Two-hour window scheduling

Task: 1 kW for 2 hours = 2 kWh. Must start at a half-hour boundary and finish within the same UTC day. The window is chosen using forecast values only.

In [ ]:
TASK_POWER_KW = 1.0
WINDOW_SLOTS = 4
INTERVAL_HOURS = 0.5


def build_window_candidates(forecasts, power_kw=1.0, slots=4, interval_hours=0.5):
    """Rank every candidate window for a constant-power task.

    Works for a single day (48 rows) or a multi-day horizon (any number of
    contiguous half-hour rows). The input is sorted internally so callers do
    not need to pre-sort.
    """
    ordered = forecasts.sort_values("timestamp_utc").reset_index(drop=True)
    assert len(ordered) >= slots
    assert ordered["timestamp_utc"].diff().dropna().eq(pd.Timedelta(minutes=30)).all()
    assert 1 <= slots <= len(ordered) and power_kw > 0
    values = ordered["predicted_gco2_per_kwh"].to_numpy()
    assert np.isfinite(values).all() and (values >= 0).all()
    energy_kwh = power_kw * interval_hours * slots
    rows = []
    for start in range(len(ordered) - slots + 1):
        window = ordered.iloc[start:start + slots]
        rows.append({
            "window_start_utc": window["timestamp_utc"].iloc[0],
            "window_end_utc": window["timestamp_utc"].iloc[-1] + pd.Timedelta(minutes=30),
            "energy_kwh": energy_kwh,
            "predicted_mean_gco2_per_kwh": window["predicted_gco2_per_kwh"].mean(),
            "predicted_emissions_gco2": window["predicted_gco2_per_kwh"].sum() * power_kw * interval_hours,
        })
    return (
        pd.DataFrame(rows)
        .sort_values(["predicted_emissions_gco2", "window_start_utc"])
        .reset_index(drop=True)
    )


# First-day demo
forecast_day = test_prediction_rows["forecast_origin_utc"].min()
day_forecasts = (
    test_prediction_rows.loc[test_prediction_rows["forecast_origin_utc"].eq(forecast_day), ["timestamp_utc", selected_method]]
    .rename(columns={selected_method: "predicted_gco2_per_kwh"}).copy()
)
window_candidates = build_window_candidates(day_forecasts, power_kw=TASK_POWER_KW, slots=WINDOW_SLOTS)
window_candidates["forecast_method"] = selected_method
selected_window = window_candidates.iloc[[0]].copy()
window_candidates.to_csv(RESULTS_DIR / "first_day_window_candidates.csv", index=False)
selected_window.to_csv(RESULTS_DIR / "first_day_selected_window.csv", index=False)
print("Forecast day:", forecast_day.date())
print("Candidate windows:", len(window_candidates))
show_table(selected_window.round(3))

## 11. Daily scheduling evaluation across all 61 test days

In [ ]:
daily_rows = []
for origin in sorted(test_prediction_rows["forecast_origin_utc"].unique()):
    day_forecasts = (
        test_prediction_rows.loc[
            test_prediction_rows["forecast_origin_utc"].eq(origin),
            ["timestamp_utc", "actual_gco2_per_kwh", selected_method],
        ]
        .rename(columns={selected_method: "predicted_gco2_per_kwh"})
        .sort_values("timestamp_utc").reset_index(drop=True)
    )
    assert len(day_forecasts) == 48
    candidates = build_window_candidates(day_forecasts[["timestamp_utc", "predicted_gco2_per_kwh"]], TASK_POWER_KW, WINDOW_SLOTS)
    selected = candidates.iloc[0]
    selected_actuals = day_forecasts.loc[
        (day_forecasts["timestamp_utc"] >= selected["window_start_utc"])
        & (day_forecasts["timestamp_utc"] < selected["window_end_utc"]),
        "actual_gco2_per_kwh",
    ]
    assert len(selected_actuals) == WINDOW_SLOTS
    actual_emissions = selected_actuals.sum() * TASK_POWER_KW * INTERVAL_HOURS

    actual_values = day_forecasts["actual_gco2_per_kwh"].to_numpy()
    sums = np.convolve(actual_values, np.ones(WINDOW_SLOTS), mode="valid")
    win_emissions = sums * TASK_POWER_KW * INTERVAL_HOURS
    oracle_idx = int(np.argmin(win_emissions))
    oracle_start = day_forecasts["timestamp_utc"].iloc[oracle_idx]
    oracle_end = day_forecasts["timestamp_utc"].iloc[oracle_idx + WINDOW_SLOTS - 1] + pd.Timedelta(minutes=30)
    oracle_emissions = float(win_emissions[oracle_idx])
    avg_window = float(win_emissions.mean())

    daily_rows.append({
        "forecast_origin_utc": origin,
        "window_start_utc": selected["window_start_utc"],
        "window_end_utc": selected["window_end_utc"],
        "predicted_mean_gco2_per_kwh": selected["predicted_mean_gco2_per_kwh"],
        "predicted_emissions_gco2": selected["predicted_emissions_gco2"],
        "actual_mean_gco2_per_kwh": selected_actuals.mean(),
        "actual_emissions_gco2": actual_emissions,
        "oracle_start_utc": oracle_start, "oracle_end_utc": oracle_end,
        "oracle_emissions_gco2": oracle_emissions,
        "average_window_emissions_gco2": avg_window,
        "emissions_error_gco2": actual_emissions - selected["predicted_emissions_gco2"],
        "abs_emissions_error_gco2": abs(actual_emissions - selected["predicted_emissions_gco2"]),
        "savings_vs_average_gco2": avg_window - actual_emissions,
        "regret_vs_oracle_gco2": actual_emissions - oracle_emissions,
    })

daily_schedule = pd.DataFrame(daily_rows)
daily_schedule.to_csv(RESULTS_DIR / "daily_window_selection.csv", index=False)
mean_selected = daily_schedule["actual_emissions_gco2"].mean()
mean_oracle = daily_schedule["oracle_emissions_gco2"].mean()
mean_average = daily_schedule["average_window_emissions_gco2"].mean()
mean_savings = daily_schedule["savings_vs_average_gco2"].mean()
mean_regret = daily_schedule["regret_vs_oracle_gco2"].mean()
mean_abs_error = daily_schedule["abs_emissions_error_gco2"].mean()
beat_average_pct = (daily_schedule["savings_vs_average_gco2"] > 0).mean() * 100

print("Days evaluated:", len(daily_schedule))
print(f"Mean selected actual emissions: {mean_selected:.2f} gCO2")
print(f"Mean oracle emissions:          {mean_oracle:.2f} gCO2")
print(f"Mean average-window emissions:  {mean_average:.2f} gCO2")
print(f"Mean savings vs average:        {mean_savings:.2f} gCO2")
print(f"Mean regret vs oracle:          {mean_regret:.2f} gCO2")
print(f"Mean absolute emission error:   {mean_abs_error:.2f} gCO2")
print(f"Days selected window beat average: {beat_average_pct:.1f}%")

In [ ]:
daily_schedule["date"] = pd.to_datetime(daily_schedule["forecast_origin_utc"]).dt.date
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(daily_schedule["date"], daily_schedule["actual_emissions_gco2"], color="#2563EB", linewidth=1.8, marker="o", markersize=3, label="Forecast-selected window")
axes[0].plot(daily_schedule["date"], daily_schedule["average_window_emissions_gco2"], color="#64748B", linewidth=1.4, linestyle="--", label="Average of all windows")
axes[0].plot(daily_schedule["date"], daily_schedule["oracle_emissions_gco2"], color="#0F766E", linewidth=1.4, linestyle=":", label="Oracle (best possible)")
axes[0].set_ylabel("Emissions for 2 kWh task (gCO2)"); axes[0].set_ylim(bottom=0)
axes[0].grid(alpha=0.2); axes[0].legend(loc="upper left", frameon=False, ncol=3)

axes[1].axhline(0, color="#94A3B8", linewidth=0.8)
axes[1].bar(daily_schedule["date"], daily_schedule["savings_vs_average_gco2"], color="#2563EB", alpha=0.7, label="Savings vs average")
axes[1].bar(daily_schedule["date"], -daily_schedule["regret_vs_oracle_gco2"], color="#0F766E", alpha=0.5, label="Regret vs oracle (inverted)")
axes[1].set_ylabel("gCO2 (positive = better)"); axes[1].set_xlabel("Test day")
axes[1].grid(alpha=0.2); axes[1].legend(loc="upper left", frameon=False, ncol=2)
for ax in axes:
    for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.suptitle("GridBrief: daily two-hour window scheduling performance", fontsize=15, fontweight="bold")
fig.text(0.02, 0.02, f"Source: NESO Carbon Intensity API | {len(daily_schedule)} test days | Mean savings vs average: {mean_savings:.1f} gCO2 ({beat_average_pct:.0f}% of days beat average)", fontsize=9, color="#64748B")
fig.tight_layout(rect=[0, 0.05, 1, 0.94])
chart_path = IMAGES_DIR / "06_daily_window_scheduling.png"
fig.savefig(chart_path, dpi=200, bbox_inches="tight"); plt.show(); plt.close(fig)
print("Chart saved:", chart_path)

## 12. Baseline comparison: does the forecast beat a fixed rule?

In [ ]:
FIXED_WINDOWS = {
    "Fixed 02:00-04:00 (overnight off-peak)": 2,
    "Fixed 12:00-14:00 (midday solar peak)": 12,
    "Fixed 18:00-20:00 (evening peak)": 18,
}
baseline_rows = []
for origin in sorted(test_prediction_rows["forecast_origin_utc"].unique()):
    day_actuals = (
        test_prediction_rows.loc[test_prediction_rows["forecast_origin_utc"].eq(origin), ["timestamp_utc", "actual_gco2_per_kwh"]]
        .sort_values("timestamp_utc").reset_index(drop=True)
    )
    assert len(day_actuals) == 48
    row = {"forecast_origin_utc": origin}
    day_start = pd.Timestamp(origin).floor("D")
    for label, start_hour in FIXED_WINDOWS.items():
        start = day_start + pd.Timedelta(hours=start_hour)
        end = start + pd.Timedelta(hours=2)
        mask = (day_actuals["timestamp_utc"] >= start) & (day_actuals["timestamp_utc"] < end)
        window = day_actuals.loc[mask, "actual_gco2_per_kwh"]
        assert len(window) == WINDOW_SLOTS
        row[label] = window.sum() * TASK_POWER_KW * INTERVAL_HOURS
    baseline_rows.append(row)

baselines = pd.DataFrame(baseline_rows)
comparison = daily_schedule.merge(baselines, on="forecast_origin_utc", how="left")
assert comparison[list(FIXED_WINDOWS)].notna().all().all()

summary_rows = [("Forecast-selected (this work)", comparison["actual_emissions_gco2"].mean())]
for label in FIXED_WINDOWS: summary_rows.append((label, comparison[label].mean()))
summary_rows.append(("Average of all 45 windows", comparison["average_window_emissions_gco2"].mean()))
summary_rows.append(("Oracle (best possible)", comparison["oracle_emissions_gco2"].mean()))

summary = (
    pd.DataFrame(summary_rows, columns=["Method", "Mean daily emissions (gCO2)"])
    .sort_values("Mean daily emissions (gCO2)").reset_index(drop=True)
)
offpeak_label = "Fixed 02:00-04:00 (overnight off-peak)"
forecast_mean = comparison["actual_emissions_gco2"].mean()
offpeak_mean = comparison[offpeak_label].mean()
forecast_vs_offpeak = offpeak_mean - forecast_mean
days_beat_offpeak = (comparison["actual_emissions_gco2"] < comparison[offpeak_label]).sum()
comparison.to_csv(RESULTS_DIR / "daily_baseline_comparison.csv", index=False)

show_table(summary.round(2))
print(f"Forecast mean:                {forecast_mean:.2f} gCO2")
print(f"Fixed off-peak mean:          {offpeak_mean:.2f} gCO2")
print(f"Forecast savings vs off-peak: {forecast_vs_offpeak:.2f} gCO2 ({forecast_vs_offpeak / offpeak_mean * 100:.1f}%)")
print(f"Days forecast beat off-peak:  {days_beat_offpeak} of {len(comparison)}")

## 13. Ridge as an alternative forecast for the scheduler

The validation step picked the persistence baseline, but Ridge had the lowest test MAE. This checks whether using Ridge as the scheduler's forecast changes the conclusion.

In [ ]:
ALT_METHOD = "Ridge Regression"
assert ALT_METHOD in test_prediction_rows.columns

alt_rows = []
for origin in sorted(test_prediction_rows["forecast_origin_utc"].unique()):
    day_forecasts = (
        test_prediction_rows.loc[
            test_prediction_rows["forecast_origin_utc"].eq(origin),
            ["timestamp_utc", "actual_gco2_per_kwh", ALT_METHOD],
        ]
        .rename(columns={ALT_METHOD: "predicted_gco2_per_kwh"})
        .sort_values("timestamp_utc").reset_index(drop=True)
    )
    assert len(day_forecasts) == 48
    candidates = build_window_candidates(day_forecasts[["timestamp_utc", "predicted_gco2_per_kwh"]], TASK_POWER_KW, WINDOW_SLOTS)
    selected = candidates.iloc[0]
    selected_actuals = day_forecasts.loc[
        (day_forecasts["timestamp_utc"] >= selected["window_start_utc"])
        & (day_forecasts["timestamp_utc"] < selected["window_end_utc"]),
        "actual_gco2_per_kwh",
    ]
    assert len(selected_actuals) == WINDOW_SLOTS
    alt_rows.append({
        "forecast_origin_utc": origin,
        "window_start_utc": selected["window_start_utc"],
        "predicted_emissions_gco2": selected["predicted_emissions_gco2"],
        "actual_emissions_gco2": selected_actuals.sum() * TASK_POWER_KW * INTERVAL_HOURS,
    })

alt_schedule = pd.DataFrame(alt_rows).merge(baselines, on="forecast_origin_utc", how="left")
alt_mean = alt_schedule["actual_emissions_gco2"].mean()
offpeak_mean = alt_schedule[offpeak_label].mean()
baseline_mean = daily_schedule["actual_emissions_gco2"].mean()
oracle_mean = daily_schedule["oracle_emissions_gco2"].mean()
alt_vs_offpeak = offpeak_mean - alt_mean
days_alt_beat_offpeak = (alt_schedule["actual_emissions_gco2"] < alt_schedule[offpeak_label]).sum()
alt_schedule.to_csv(RESULTS_DIR / "daily_window_selection_ridge.csv", index=False)

headline = pd.DataFrame([
    ("Oracle (best possible)", oracle_mean),
    ("Fixed 02:00-04:00 off-peak", offpeak_mean),
    (f"Forecast-selected: {ALT_METHOD}", alt_mean),
    ("Forecast-selected: Same time yest.", baseline_mean),
    ("Average of all 45 windows", daily_schedule["average_window_emissions_gco2"].mean()),
], columns=["Method", "Mean daily emissions (gCO2)"]).sort_values("Mean daily emissions (gCO2)").reset_index(drop=True)
show_table(headline.round(2))
print(f"Ridge forecast mean:            {alt_mean:.2f} gCO2")
print(f"Off-peak rule mean:             {offpeak_mean:.2f} gCO2")
print(f"Ridge savings vs off-peak:      {alt_vs_offpeak:.2f} gCO2 ({alt_vs_offpeak / offpeak_mean * 100:+.1f}%)")
print(f"Days Ridge beat off-peak:       {days_alt_beat_offpeak} of {len(alt_schedule)}")
print(f"Ridge vs Same-time-yesterday:   {baseline_mean - alt_mean:+.2f} gCO2")

In [ ]:
methods = [
    ("Oracle (best possible)", daily_schedule["oracle_emissions_gco2"], "#0F766E"),
    (offpeak_label, baselines[offpeak_label], "#14B8A6"),
    ("Ridge forecast-selected", alt_schedule["actual_emissions_gco2"], "#2563EB"),
    ("Same-time-yesterday forecast", daily_schedule["actual_emissions_gco2"], "#64748B"),
    ("Average random window", daily_schedule["average_window_emissions_gco2"], "#94A3B8"),
]
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [2, 1]})
ax = axes[0]
for i, (label, values, color) in enumerate(methods):
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(values))
    ax.scatter(values, np.full(len(values), i) + jitter, color=color, alpha=0.55, s=22)
    ax.scatter([values.mean()], [i], color=color, s=180, marker="D", edgecolor="white", linewidth=1.5, zorder=5)
ax.set_yticks(range(len(methods))); ax.set_yticklabels([m[0] for m in methods], fontsize=10)
ax.invert_yaxis(); ax.set_xlabel("Emissions for 2 kWh task (gCO2); lower is better")
ax.grid(axis="x", alpha=0.2)
for s in ("top", "right"): ax.spines[s].set_visible(False)

ax = axes[1]
forecast_methods = [
    ("Ridge", alt_schedule["actual_emissions_gco2"] < alt_schedule[offpeak_label]),
    ("Same time yesterday", daily_schedule["actual_emissions_gco2"] < baselines[offpeak_label]),
]
labels = [m[0] for m in forecast_methods]
pcts = [m[1].mean() * 100 for m in forecast_methods]
colors = ["#2563EB", "#64748B"]
bars = ax.barh(labels, pcts, color=colors, height=0.5)
ax.axvline(50, color="#94A3B8", linewidth=1, linestyle="--")
ax.text(50, -0.6, "50% = ties off-peak", ha="center", fontsize=9, color="#64748B")
ax.set_xlim(0, 100); ax.set_xlabel("% of 61 test days beating off-peak rule")
ax.bar_label(bars, fmt="%.0f%%", padding=5, fontsize=11)
ax.grid(axis="x", alpha=0.2)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.invert_yaxis()
fig.suptitle("GridBrief: forecast scheduling does not beat overnight off-peak", fontsize=15, fontweight="bold")
fig.text(0.02, 0.02, f"Source: NESO Carbon Intensity API | {len(daily_schedule)} test days | Task: 1 kW for 2 h = 2 kWh | Overnight off-peak mean: {baselines[offpeak_label].mean():.1f} gCO2", fontsize=9, color="#64748B")
fig.tight_layout(rect=[0, 0.05, 1, 0.93])
chart_path = IMAGES_DIR / "07_baseline_comparison.png"
fig.savefig(chart_path, dpi=200, bbox_inches="tight"); plt.show(); plt.close(fig)
print("Chart saved:", chart_path)

## 14. Seven-day scheduling horizon

The natural follow-up: give the scheduler a whole week to place the task. Does the wider horizon rescue the forecast?

In [ ]:
ALT_METHOD = "Ridge Regression"
HORIZON_DAYS = 7
all_origins = sorted(test_prediction_rows["forecast_origin_utc"].unique())

week_rows = []
for i in range(0, len(all_origins) - HORIZON_DAYS + 1, HORIZON_DAYS):
    week_origins = all_origins[i:i + HORIZON_DAYS]
    week_start, week_end = week_origins[0], week_origins[-1] + pd.Timedelta(days=1)
    week_data = (
        test_prediction_rows.loc[
            test_prediction_rows["forecast_origin_utc"].isin(week_origins),
            ["timestamp_utc", "actual_gco2_per_kwh", ALT_METHOD],
        ]
        .rename(columns={ALT_METHOD: "predicted_gco2_per_kwh"})
        .sort_values("timestamp_utc").reset_index(drop=True)
    )
    candidates = build_window_candidates(week_data[["timestamp_utc", "predicted_gco2_per_kwh"]], TASK_POWER_KW, WINDOW_SLOTS)
    selected = candidates.iloc[0]
    selected_actuals = week_data.loc[
        (week_data["timestamp_utc"] >= selected["window_start_utc"])
        & (week_data["timestamp_utc"] < selected["window_end_utc"]),
        "actual_gco2_per_kwh",
    ]
    assert len(selected_actuals) == WINDOW_SLOTS
    actual_emissions = selected_actuals.sum() * TASK_POWER_KW * INTERVAL_HOURS

    actuals = week_data["actual_gco2_per_kwh"].to_numpy()
    win_sums = np.convolve(actuals, np.ones(WINDOW_SLOTS), mode="valid")
    win_emissions = win_sums * TASK_POWER_KW * INTERVAL_HOURS
    oracle_idx = int(win_emissions.argmin())
    oracle_emissions = float(win_emissions[oracle_idx])
    oracle_start = week_data["timestamp_utc"].iloc[oracle_idx]

    fixed_start = week_start + pd.Timedelta(hours=2)
    fixed_end = fixed_start + pd.Timedelta(hours=2)
    fixed_mask = (week_data["timestamp_utc"] >= fixed_start) & (week_data["timestamp_utc"] < fixed_end)
    fixed_emissions = week_data.loc[fixed_mask, "actual_gco2_per_kwh"].sum() * TASK_POWER_KW * INTERVAL_HOURS

    week_rows.append({
        "week_start_utc": week_start, "week_end_utc": week_end,
        "selected_start_utc": selected["window_start_utc"],
        "selected_actual_emissions_gco2": actual_emissions,
        "oracle_start_utc": oracle_start, "oracle_emissions_gco2": oracle_emissions,
        "average_window_emissions_gco2": float(win_emissions.mean()),
        "fixed_monday_offpeak_emissions_gco2": fixed_emissions,
    })

weekly = pd.DataFrame(week_rows)
weekly.to_csv(RESULTS_DIR / "weekly_window_selection.csv", index=False)
sel_mean = weekly["selected_actual_emissions_gco2"].mean()
fix_mean = weekly["fixed_monday_offpeak_emissions_gco2"].mean()
orc_mean = weekly["oracle_emissions_gco2"].mean()
avg_mean = weekly["average_window_emissions_gco2"].mean()
beats_fixed = (weekly["selected_actual_emissions_gco2"] < weekly["fixed_monday_offpeak_emissions_gco2"]).sum()

summary = pd.DataFrame([
    ("Oracle (best window in week)", orc_mean),
    ("Fixed Monday 02:00-04:00", fix_mean),
    (f"Forecast-selected (Ridge), {HORIZON_DAYS}-day horizon", sel_mean),
    ("Average of all windows", avg_mean),
], columns=["Method", "Mean weekly emissions (gCO2)"]).sort_values("Mean weekly emissions (gCO2)").reset_index(drop=True)
show_table(summary.round(2))
print(f"Weeks evaluated:               {len(weekly)}")
print(f"Forecast-selected mean:        {sel_mean:.2f} gCO2")
print(f"Fixed Monday off-peak mean:    {fix_mean:.2f} gCO2")
print(f"Savings vs fixed rule:         {fix_mean - sel_mean:+.2f} gCO2 ({(fix_mean - sel_mean) / fix_mean * 100:+.1f}%)")
print(f"Weeks forecast beat fixed:     {beats_fixed} of {len(weekly)}")
print(f"Savings vs average window:     {avg_mean - sel_mean:+.2f} gCO2")
print(f"Regret vs oracle:              {sel_mean - orc_mean:+.2f} gCO2")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [2, 1]})
ax = axes[0]
weeks = np.arange(1, len(weekly) + 1)
ax.plot(weeks, weekly["oracle_emissions_gco2"], color="#0F766E", marker="o", linewidth=1.8, label="Oracle (best window in week)")
ax.plot(weeks, weekly["fixed_monday_offpeak_emissions_gco2"], color="#14B8A6", marker="s", linewidth=1.8, linestyle="--", label="Fixed Monday 02:00-04:00")
ax.plot(weeks, weekly["selected_actual_emissions_gco2"], color="#2563EB", marker="D", linewidth=1.8, label="Ridge forecast-selected (7-day horizon)")
ax.plot(weeks, weekly["average_window_emissions_gco2"], color="#94A3B8", marker="^", linewidth=1.4, linestyle=":", label="Average random window")
ax.set_xticks(weeks); ax.set_xticklabels([f"W{i}" for i in weeks])
ax.set_xlabel("Test week"); ax.set_ylabel("Emissions for 2 kWh task (gCO2)")
ax.set_ylim(bottom=0); ax.grid(alpha=0.2)
ax.legend(loc="upper left", frameon=False, fontsize=9)
for s in ("top", "right"): ax.spines[s].set_visible(False)

ax = axes[1]
rows = [
    ("Oracle", weekly["oracle_emissions_gco2"].mean(), "#0F766E"),
    ("Fixed Monday\n02:00-04:00", weekly["fixed_monday_offpeak_emissions_gco2"].mean(), "#14B8A6"),
    ("Ridge\nforecast (7-day)", weekly["selected_actual_emissions_gco2"].mean(), "#2563EB"),
    ("Average\nrandom window", weekly["average_window_emissions_gco2"].mean(), "#94A3B8"),
]
order = np.argsort([r[1] for r in rows])
labels = [rows[i][0] for i in order]
vals = [rows[i][1] for i in order]
colors = [rows[i][2] for i in order]
bars = ax.barh(labels, vals, color=colors, height=0.55)
ax.bar_label(bars, fmt="%.0f", padding=5, fontsize=11)
ax.set_xlabel("Mean weekly emissions (gCO2)")
ax.set_xlim(0, max(vals) * 1.2); ax.grid(axis="x", alpha=0.2)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.invert_yaxis()

fig.suptitle("GridBrief: 7-day scheduling horizon still loses to a fixed rule", fontsize=15, fontweight="bold")
fig.text(0.02, 0.02, f"Source: NESO Carbon Intensity API | {len(weekly)} test weeks | Task: 1 kW for 2 h = 2 kWh | Ridge beat fixed rule on {beats_fixed} of {len(weekly)} weeks", fontsize=9, color="#64748B")
fig.tight_layout(rect=[0, 0.05, 1, 0.93])
chart_path = IMAGES_DIR / "08_weekly_horizon_comparison.png"
fig.savefig(chart_path, dpi=200, bbox_inches="tight"); plt.show(); plt.close(fig)
print("Chart saved:", chart_path)

## Summary

- **Ridge Regression** had the lowest test MAE (**40.83 gCO2/kWh**).
- Validation selected **same time yesterday**; the test set exposed that choice as suboptimal.
- On the daily and 7-day horizons, **no forecast method beat a trivial fixed 02:00-04:00 off-peak rule**.
- The negative result is the point: on a grid with a strong structural low-carbon window, a forecast only beats the trivial rule if it can out-predict the prior that rule already encodes.

**Charts produced** (in `/content/gridbrief/images/`):
- `01_daily_carbon_intensity.png`
- `02_monthly_distribution.png`
- `03_weekday_time_heatmap.png`
- `04_test_model_comparison.png`
- `05_first_test_week_forecast.png`
- `06_daily_window_scheduling.png`
- `07_baseline_comparison.png`
- `08_weekly_horizon_comparison.png`

**Data artifacts** (in `/content/gridbrief/results/`): test predictions, daily and weekly scheduling tables, baseline comparison, model selection record, and validation metrics.

**Why off-peak wins.** GB demand is lowest between 02:00 and 04:00, and overnight wind is often curtailed or exported rather than displacing gas. The grid operator's own published intensity series already reflects that structural pattern, so a forecast has to beat a rule that encodes it. Ridge and the persistence baseline sometimes catch a windy night better than the fixed window, but not often enough across Nov and Dec to shift the daily mean.

**What would change the result.** A longer test period including spring and summer (more solar, weaker overnight advantage), weather-driven features (wind speed, demand), or a longer-duration task where the window matters more per kWh. None of those are in scope here.